In [22]:
import re
import pandas as pd

# 샘플: 갑구/을구 테이블 형태를 가정한 DataFrame
sample_gap = pd.DataFrame({
    "순위번호": [10, 4, 5, 7, 8],
    "등기목적": ["압류", "압류", "가압류", "가압류", "가압류"],
    "접수정보": [
        "2024년10월22일 제182334호",
        "2024년8월14일 제142780호",
        "2024년10월4일 제173509호",
        "2024년10월10일 제175994호",
        "2024년10월16일 제179733호",
    ],
    "주요등기사항": [
        "권리자 국민건강보험공단",
        "권리자 국",
        "청구금액 금39,352,737원 채권자 주식회사씨앤제이헬스케어",
        "청구금액 금810,000,000 원 채권자 신용보증기금",
        "청구금액 금456,486,258 원 채권자 주식회사애큐온캐피탈",
    ],
    "대상소유자": ["신명주", "신명주", "신명주", "신명주", "신명주"],
})

sample_eul = pd.DataFrame({
    "순위번호": [10],
    "등기목적": ["근저당권설정"],
    "접수정보": ["2017년6월15일 제28906호"],
    "주요등기사항": ["채권최고액 금826,800,000원 근저당권자 주식회사신한은행"],
    "대상소유자": ["남상길"],
})

# 1) 날짜 추출 (접수정보)
_date_pat = re.compile(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일")

def extract_date(s: str) -> str:
    if not s:
        return ""
    m = _date_pat.search(str(s))
    if not m:
        return ""
    y, mth, d = m.groups()
    return f"{y}-{int(mth):02d}-{int(d):02d}"

# 2) 권리자/채권자/근저당권자 추출 (주요등기사항)
_creditor_pats = [
    re.compile(r"권리자\s*([\w\d가-힣()·.,&\s]+)"),
    re.compile(r"채권자\s*([\w\d가-힣()·.,&\s]+)"),
    re.compile(r"근저당권자\s*([\w\d가-힣()·.,&\s]+)"),
]

def extract_creditor(s: str) -> str:
    text = str(s or "")
    for pat in _creditor_pats:
        m = pat.search(text)
        if m:
            return m.group(1).strip()
    return ""

# 3) 청구금액/채권최고액 추출 (주요등기사항)
_amount_pats = [
    re.compile(r"청구금액\s*금\s*([0-9,]+)\s*원"),
    re.compile(r"채권최고액\s*금\s*([0-9,]+)\s*원"),
]

def extract_amount(s: str) -> str:
    text = str(s or "")
    for pat in _amount_pats:
        m = pat.search(text)
        if m:
            return m.group(1).replace(",", "")
    return ""

# 파싱 적용 - 최종 컬럼 순서: 순위번호, 등기목적, 접수정보(날짜), 채권자/권리자, 청구금액, 비고, 임금채권추정, 대상소유자

def parse_gap_eul(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["순위번호", "등기목적", "접수정보", "채권자/권리자", "청구금액", "비고", "임금채권추정", "대상소유자"])
    
    out = df.copy()
    
    # 기존 컬럼 유지 및 새 컬럼 추가
    out["접수정보"] = out.get("접수정보", "").apply(extract_date)  # 날짜로 변환
    out["채권자/권리자"] = out.get("주요등기사항", "").apply(extract_creditor)
    out["청구금액"] = out.get("주요등기사항", "").apply(extract_amount)
    
    # 빈 컬럼 추가
    out["비고"] = ""
    out["임금채권추정"] = ""
    out["지번번호"] = ""
    
    # 대상소유자 컬럼이 없으면 빈 값으로 채우기
    if "대상소유자" not in out.columns:
        out["대상소유자"] = ""
    
    # 등기목적 기준 정렬: "가압류", "압류" 다음 나머지들
    def sort_key(x):
        if x == "가압류":
            return 0
        elif x == "압류":
            return 1
        else:
            return 2
    
    out["_sort_key"] = out["등기목적"].apply(sort_key)
    out = out.sort_values(["_sort_key", "순위번호"]).drop(columns=["_sort_key"])
    
    # 등기목적별로 순위번호 재매기기 (1부터 시작)
    out["순위번호"] = out.groupby("등기목적").cumcount() + 1
    
    # 최종 컬럼 순서로 정렬
    final_cols = ["순위번호", "등기목적", "접수정보", "채권자/권리자", "청구금액", "비고", "임금채권추정", "대상소유자", "지번번호"]
    out = out.reindex(columns=final_cols)
    
    return out

In [23]:
parse_gap_eul(sample_gap)


,순위번호,등기목적,접수정보,채권자/권리자,청구금액,비고,임금채권추정,대상소유자,지번번호
2,1,가압류,2024-10-04,주식회사씨앤제이헬스케어,39352737,,,신명주,
3,2,가압류,2024-10-10,신용보증기금,810000000,,,신명주,
4,3,가압류,2024-10-16,주식회사애큐온캐피탈,456486258,,,신명주,
1,1,압류,2024-08-14,국,,,,신명주,
0,2,압류,2024-10-22,국민건강보험공단,,,,신명주,


In [10]:
# Datadisk 유틸 실행 예시
import pandas as pd
from utils import Datadisk


In [11]:
# 경로 설정 (필요시 수정)
excel_path = "./data/datadisk/KB_2024_5.xlsx"
cfg_path = "./cfg/extract_columns.json"


In [12]:
# 실행
dd = Datadisk()
df = dd.run(excel_path=excel_path, cfg_path=cfg_path)
# 결과 확인
df.head()


AttributeError: 'str' object has no attribute 'astype'